# Stage 01 — Review (T1 + T3 + M1/M2/M3)

For every patent in `matched/` (output of Stage 00b2 — figures already cropped
and matched to BRIEF DESCRIPTION lines):
1. Read figure label from filename (Stage 00b2 encoded it; no re-OCR)
2. Read BRIEF DESCRIPTION from `text/<patent_id>.txt`
3. Match filename label → description line → assign `match_status`
4. SigLIP zero-shot: T2 (per-figure visual fields), G1 (architecture topType),
   M1 (fuselage/gear/symmetry), M2 (wing config/empennage), M3 (propulsion)
5. PatentSBERTa: T1 dimensions (scope, field, target)
6. Write `labels/<patent_id>.json`
7. Generate per-patent review HTML pre-filled with all AI predictions

SigLIP and PatentSBERTa are **required** — the notebook will not run without them.

**match_status colour key:**
- 🟢 `matched` — filename label found and matched to a description line (confidence 0.95)
- 🔴 `unmatched` — filename label found but no matching line in description text (confidence 0.0)
- 🟡 `no_label` — no label in filename; positional fallback used (confidence 0.3)
- 🟠 `duplicate` — same figure number claimed by multiple images (confidence 0.9)

Rows with `needs_review: true` are **highlighted with a yellow border**.

In [ ]:
import sys
from pathlib import Path

_cwd = Path().resolve()
repo_root = None
for _candidate in [_cwd, *_cwd.parents]:
    if (_candidate / "src").exists() and (_candidate / "config.yaml").exists():
        repo_root = _candidate
        break
if repo_root is None:
    raise RuntimeError(f"Cannot find repo root from {_cwd}. Run from inside Patent-Labelling-Tools.")

for p in [str(repo_root), str(repo_root / "src")]:
    while p in sys.path:
        sys.path.remove(p)
sys.path.insert(0, str(repo_root / "src"))
sys.path.insert(0, str(repo_root))
print(f"repo_root : {repo_root}")


In [ ]:
import os
from src.config_loader import load_config
cfg = load_config()

# Must run BEFORE any sentence_transformers/open_clip/huggingface_hub import:
# huggingface_hub freezes HF_HUB_CACHE as a constant at first import, so
# setting it afterward is silently ignored and weights re-download to the
# default ~/.cache/huggingface instead of the project models/ folder.
os.environ["HF_HUB_CACHE"] = str(cfg["paths"]["siglip_cache"])
os.environ["HF_HOME"]      = str(cfg["paths"]["siglip_cache"])

from src.matcher import parse_description, match_images, label_from_filename
from src.cross_modal import load_siglip_model, verify_matches

# ── PatentSBERTa — required for T1 dimension classification ──────────────────
# Cached under cfg["paths"]["sbert_cache"] (Patent-Labelling-Tools/models/SBERT)
# so weights stay inside the project instead of ~/.cache/huggingface.
from sentence_transformers import SentenceTransformer
sbert = SentenceTransformer("AI-Growth-Lab/PatentSBERTa", cache_folder=str(cfg["paths"]["sbert_cache"]))
print("PatentSBERTa ready.")

# ── SigLIP — required for T2/G1/M1/M2/M3 visual classification ───────────────
USE_SIGLIP = True
siglip_bundle = load_siglip_model(cache_dir=cfg["paths"]["siglip_cache"])
print(f"SigLIP ready on {siglip_bundle[3]}.")


In [ ]:
# ── Batch selector ────────────────────────────────────────────────────────
# Set BATCH_ID to the batch you want to review (see data/batches.xlsx Summary sheet).
BATCH_ID = 0

import pandas as pd
from pathlib import Path

cfg = __import__("src.config_loader", fromlist=["load_config"]).load_config()
batches_path = Path(cfg["paths"]["data"]) / "batches.xlsx"

if not batches_path.exists():
    raise FileNotFoundError(f"batches.xlsx not found at {batches_path} — run 00b1_grouping first.")

sheet_name  = f"Batch_{BATCH_ID:02d}"
batch_df    = pd.read_excel(batches_path, sheet_name=sheet_name, dtype=str)
patent_ids  = batch_df["patent_id"].dropna().str.strip().tolist()

print(f"Batch {BATCH_ID}: {len(patent_ids)} patents loaded from {sheet_name}")
print(batch_df[["patent_id","company_canonical","prototype_label"]].head(10).to_string(index=False))


In [ ]:
from src.config_loader import load_config
from src.extractor import load_patseer_excel
from src.reviewer import process_patent

cfg = load_config()
matched_dir = cfg['paths']['matched']
text_dir    = cfg['paths']['text']
labels_dir  = cfg['paths']['labels']
print(f"matched : {matched_dir}")
print(f"text    : {text_dir}")
print(f"labels  : {labels_dir}")


In [ ]:
# Excel used for T1 metadata only (title, assignee, years, citations).
# Drawing descriptions are read from text/<patent_id>.txt by process_patent().
excel_index = load_patseer_excel(cfg['paths']['patseer_excel'])
print(f"Indexed {len(excel_index)} patents.")

In [ ]:
# ── Single-patent diagnostic (optional) ─────────────────────────────────────
# Set INSPECT_ID to any patent folder name to process just that one patent
# and see its record dict before running the full batch.
# Leave as None to skip.
INSPECT_ID = None  # e.g. "US2015014475A1"

if INSPECT_ID:
    from src.reviewer import process_patent
    data = process_patent(
        INSPECT_ID, cfg, excel_index, matched_dir,
        sbert_model=sbert, siglip_bundle=siglip_bundle, skip_siglip=not USE_SIGLIP,
    )
    print(f"Figures: {len(data.get('T3_images', []))}")
    print(f"has_splits: {data.get('has_splits')}")


In [ ]:
# ── Batch run configuration ──────────────────────────────────────────────────
LIMIT = None   # int or None — None processes all patents in matched/
# SigLIP and SBERT are always on — see model-loading cell above.
# ─────────────────────────────────────────────────────────────────────────────


In [ ]:
from src.reviewer import run_stage01

summary_df = run_stage01(
    cfg,
    sbert_model   = sbert,           # loaded in model-loading cell above
    siglip_bundle = siglip_bundle,   # None if SigLIP not loaded
    skip_siglip   = not USE_SIGLIP,
    patent_ids    = patent_ids,   # process only this batch
)

In [ ]:
print(summary_df.to_string(index=False))

In [ ]:
# ── Single-patent diagnostic — change INSPECT_ID to inspect any patent ─────────
import pandas as pd

INSPECT_ID = None   # e.g. "US2015014475A1"

if INSPECT_ID:
    review_df = pd.read_excel(cfg["paths"]["source_excel"], sheet_name="Review")
    sub = review_df[review_df["Patent_ID"] == INSPECT_ID]
    if sub.empty:
        print(f"No rows found for {INSPECT_ID} in source_patents.xlsx")
    else:
        print(f"\n{INSPECT_ID}  |  {len(sub)} rows")
        print(sub[["Section", "Sub_Dimension", "Field", "Value", "Confidence", "Source"]].to_string(index=False))


In [ ]:
# ── Copy approved files → reviewed/{company}/{prototype}/{patent_id}/ ──────
# Run this after you have finished human review for this batch.
# Only patents with human_review_status == "approved" in batches.xlsx are copied.
# Raw files are never touched. Source is matched/{patent_id}/.

import shutil
from pathlib import Path
import pandas as pd

cfg           = __import__("src.config_loader", fromlist=["load_config"]).load_config()
batches_path  = Path(cfg["paths"]["data"]) / "batches.xlsx"
matched_root  = Path(cfg["paths"]["matched"])
reviewed_root = Path(cfg["paths"]["reviewed"])
sheet_name    = f"Batch_{BATCH_ID:02d}"

batch_df = pd.read_excel(batches_path, sheet_name=sheet_name, dtype=str)
approved = batch_df[batch_df["human_review_status"].str.strip().str.lower() == "approved"]

copied  = 0
skipped = 0

for _, row in approved.iterrows():
    pid       = str(row["patent_id"]).strip()
    company   = str(row["company_canonical"]).strip()
    prototype = str(row["prototype_label"]).strip()

    src_dir  = matched_root / pid
    dest_dir = reviewed_root / company / prototype / pid

    if not src_dir.exists():
        print(f"  ⚠  matched/{pid}/ not found — skipping")
        skipped += 1
        continue

    dest_dir.mkdir(parents=True, exist_ok=True)

    for f in src_dir.glob("*.png"):
        dest_f = dest_dir / f.name
        if not dest_f.exists():
            shutil.copy2(f, dest_f)
            copied += 1
        else:
            skipped += 1

print(f"Approved patents : {len(approved)}")
print(f"Files copied     : {copied}")
print(f"Already existed  : {skipped}")
print(f"Output root      : {reviewed_root}")


In [ ]:
# ── Collect batch outputs → batch_00/ ─────────────────────────────────────────
# Copies this batch's rows from source_patents.xlsx into batch_00/ for validation.
# When you're happy and run all patents, skip this cell.

import pandas as pd
from pathlib import Path

BATCH_OUT = Path(cfg["paths"]["base"]) / "batch_00"
BATCH_OUT.mkdir(parents=True, exist_ok=True)

review_df = pd.read_excel(cfg["paths"]["source_excel"], sheet_name="Review")
batch_rows = review_df[review_df["Patent_ID"].isin(batch_df["patent_id"])]
out_path = BATCH_OUT / "source_patents.xlsx"
batch_rows.to_excel(out_path, sheet_name="Review", index=False)

print(f"Batch outputs collected → {out_path}")
print(f"  rows : {len(batch_rows)}")
